In [1]:
import sys
import os

import streamlit as st
from core.utils import Sermon, Person, get_short_info
from collections import Counter

import plotly.express as px
from plotly.subplots import make_subplots
import networkx as nx

import plotly.graph_objects as go
import pandas as pd

import json
import re

from rapidfuzz import fuzz

In [2]:
def flatten(xss):
    return [x for xs in xss for x in xs]

In [3]:
def create_quote_dist_chart(ids: list, type: str, searchkey: str="") -> go.Figure:
    type_dict = {
        "orgelpredigt": "Orgelpredigtzitate",
        "musikwerk": "Liedzitate",
        "quelle": "Literaturzitate",
    }
    if type not in type_dict.keys():
        occ_fig = go.Figure()
        occ_fig.update_layout(title_text="Type not recognised!")
        return occ_fig
    
    else:
        chunked_text = [0]*100
        thumbnails = [""]*100

        for id in ids:
            sermon = Sermon(id)

            dec = int(len(sermon.words) / 99)
            overhang = len(sermon.words) % dec

            for i, j in zip(range(0, len(sermon.words), dec), range(0, 100)):
                if searchkey != "":
                    keys_unique = sermon.reference[i:i+dec]
                    keys_str = " ".join(flatten(keys_unique))
                    if searchkey in keys_str:
                        hit_test = 1
                        hit = f"{sermon.kurztitel}"
                    else:
                        hit_test = 0
                        hit = ""
                else:
                    types_unique = list(set(sermon.word_types[i:i+dec]))
                    types_str = " ".join([x for x in types_unique if isinstance(x, str)])
                    if type in types_str:
                        hit_test = 1
                        hit = f"{sermon.kurztitel}<br>"
                    else:
                        hit_test = 0
                        hit = ""
                
                chunked_text[j] = chunked_text[j] + hit_test
                thumbnails[j] = thumbnails[j] + hit
            if searchkey != "":
                last_keys_unique = sermon.reference[-overhang:]
                last_keys_str = " ".join(flatten(last_keys_unique))
                if searchkey in last_keys_str:
                    last_hit_test = 1
                    last_hit = f"{sermon.kurztitel}<br>"
                else:
                    last_hit_test = 0
                    last_hit = ""
            else:
                last_types_unique = list(set(sermon.word_types[-overhang:]))
                last_types_str = " ".join([x for x in last_types_unique if isinstance(x, str)])
                if type in last_types_str:
                    last_hit_test = 1
                    last_hit = f"{sermon.kurztitel}<br>"
                else:
                    last_hit_test = 0
                    last_hit = ""
            
            #chunked_text[-1] = chunked_text[-1] + last_orgelpredigt_test
            #thumbnails[-1] = thumbnails[-1] + last_hit

        occ_fig = go.Figure()

        for i in range(0, len(chunked_text)):
            hovertext = f'{chunked_text[i]} {type_dict[type]} im {i+1}%'
            if thumbnails[i] != "":
                    hovertext += f"<br>{thumbnails[i]}"

            gradient = chunked_text[i] * 15
            color = f'rgb({max(250-gradient, 0)},{max(250-gradient, 0)},{max(250-gradient, 0)})'
            occ_fig.add_trace(go.Bar(
                x = [f"{type_dict[type]} je Predigtprozent"],
                y = [100],
                marker_color = color,
                hovertext = hovertext
            ))

        occ_fig.update_layout(width=1500,height=500, showlegend=False)
        print(hit)
        return occ_fig

def group_sermons_in_years(data, interval: int) -> list:
    chunked_sermons = []
    start_year = 1600
    end_year = 1800
    yearfinder = re.compile(r'[0-9]{4}')
    for i in range(start_year, end_year, interval):
        sermons = []
        for id, info in data.items():
            year = int(re.findall(yearfinder, info['year'])[0])
            if year > i and year < i + interval:
                sermons.append(id)
        chunked_sermons.append(sermons)

    return chunked_sermons

In [4]:
# Get the list of all files in a directory
with open("predigten_übersicht.json", "r", encoding="utf-8") as file: 
    data = json.load(file)

# Ensure all entries have a 'year' key
cleaned = {k: v for k, v in data.items() if 'year' in v}

year_finder = re.compile(r'[0-9]{4}')

for k, v in data.items():
    year = re.findall(year_finder, v['year'])[0]
    if year:
        v['year'] = year
    else:
        v['year'] = '[s.a.]'

# Convert to nested list and sort by year
relevant_sermons = sorted(
    [[key, value['title'], int(value['year'])] for key, value in cleaned.items()],
    key=lambda x: x[2]
)

ids = [i[0] for i in relevant_sermons]

In [5]:
praetorius = "E080223"
in_dulci = "E100017"
dieterich = "E000003"

In [6]:
quote_time_dist = "gesamter_zeitraum"
quote_type = "orgelpredigt"
searchkey = in_dulci

In [7]:
if quote_time_dist == "50-Jahr-Intervalle":
    sermons_grouped_50 = group_sermons_in_years(data, 50)
    figs_50 = []
    for i in range(len(sermons_grouped_50)):
        figs_50.append(create_quote_dist_chart(sermons_grouped_50[i], quote_type,searchkey=searchkey))
    
    # Create subplots
    fig = make_subplots(rows=len(figs_50), 
                        cols=1, 
                        subplot_titles=[f"Verteilung in Predigten zwischen {1600 + (i*50)} und {1600+(i*50)+50} ({len(sermons_grouped_50[i])} Predigten)" for i in range(len(figs_50))])

    # Add traces from each figure to the subplots
    for i, fig_item in enumerate(figs_50):
        for trace in fig_item.data:
            fig.add_trace(trace, row=i+1, col=1)

    # Update layout
    fig.update_layout(height=1200, width=1000, showlegend = False)
    fig.update_layout(title_text="Accumulierte Verteilung von Zitaten in 50-Jahr Intervallen")

elif quote_time_dist == "25-Jahr-Intervalle":
    sermons_grouped_25 = group_sermons_in_years(data, 25)
    figs_25 = []
    for i in range(len(sermons_grouped_25)):
        figs_25.append(create_quote_dist_chart(sermons_grouped_25[i], quote_type,searchkey=searchkey))
    
    # Create subplots
    fig = make_subplots(rows=len(figs_25), 
                        cols=1, 
                        subplot_titles=[f"Verteilung in Predigten zwischen {1600 + (i*25)} und {1600+(i*25)+25} ({len(sermons_grouped_25[i])} Predigten)" for i in range(len(figs_25))])

    # Add traces from each figure to the subplots
    for i, fig_item in enumerate(figs_25):
        for trace in fig_item.data:
            fig.add_trace(trace, row=i+1, col=1)

    # Update layout
    fig.update_layout(height=1200, width=1000, showlegend = False)
    fig.update_layout(title_text="Accumulierte Verteilung von Zitaten in 25-Jahr Intervallen")

else:
    fig = create_quote_dist_chart(ids, quote_type, searchkey=searchkey)

fig.show()

In [8]:
sermon = Sermon("E000036")
quoted_item = "E100022"
passages = []
for i in range(len(sermon.chunked)):
    for j in range(len(sermon.chunked[i])):
        words = sermon.chunked[i][j]["words"]
        types = sermon.chunked[i][j]["types"]
        refs = sermon.chunked[i][j]["references"]
        test_refs = []
        for nr, ref in enumerate(refs):
            if quoted_item in ref:
                test_refs.append(nr)
        if len(test_refs):
            passages.append(" ".join(words))


In [9]:
passages

['zwingt die säiten in cithara,',
 'und laßt die süsse musica gantz freudenreich erschallen,',
 'daß ich möge mit jesulein,',
 'dem wunderschönen bräutgam mein,',
 'jn steter liebe wallen.',
 'singet, springet, jubiliret, triumphiret,',
 'danckt dem herren, groß ist der könig der ehren!',
 'singet, springet, jubiliret, triumphiret,',
 'danckt dem herren, groß ist der könig der ehren.']

In [10]:
def get_quoted_passages(quoted_item: str, ids: list):
    # get passages from quoted item in every sermon
    passages = []
    for id in ids:
        sermon = Sermon(id)
        for i in range(len(sermon.chunked)):
            for j in range(len(sermon.chunked[i])):
                words = sermon.chunked[i][j]["words"]
                types = sermon.chunked[i][j]["types"]
                refs = sermon.chunked[i][j]["references"]
                test_refs = []
                for nr, ref in enumerate(refs):
                    if quoted_item in ref:
                        test_refs.append(nr)
                if len(test_refs):
                    if j > 0:
                        words_before = " ".join(sermon.chunked[i][j-1]["words"])
                    else:
                        words_before = " ".join(sermon.chunked[i-1][-1]["words"])
                    if j < len(sermon.chunked[i])-1:
                        words_after = " ".join(sermon.chunked[i][j+1]["words"])
                    else:
                        words_after = " ".join(sermon.chunked[i+1][0]["words"])
                    passages.append([id, " ".join(words), words_before, words_after])
    
    return passages

passages = get_quoted_passages("E100022", ids)

In [12]:
results = pd.DataFrame(passages, columns=['sermon', 'verse', 'text_before', 'text_after'])

In [13]:
liedtext_morgenstern = [
"Wie schön leuchtet der Morgenstern /",
"Voll Gnad vnd Warheit von dem HERRN /",
"Die süsse Wurtzel Jesse?",
"Du Sohn Dauid auß Jacobs Stamm /",
"Mein König vnd mein Bräutigam /",
"Hast mir mein Hertz besessen /",
"Lieblich freundtlich Schön vnd herrlich /",
"Groß vnd ehrlich Reich von Gaben /",
"Hoch vnd sehr prächtig erhaben.",
"Ey mein Perle / du werthe Kron /",
"Wahr Gottes vnd Marien Sohn /",
"Ein hochgeborner König /",
"Mein Hertz heißt dich ein lilium,",
"Dein süsses Euangelium,",
"Jst lauter Milch vnd Honig /",
"Ey mein Blümlein Hosianna /",
"Himmlisch Manna Das wir essen /",
"Deiner kan ich nicht vergessen.",
"Geuß sehr tieff in mein Hertz hineyn /",
"Du heller Jaspis vnd Rubin /",
"Die Flamme deiner Liebe.",
"Vnd erfreuw mich / daß ich doch bleib",
"An deinem außerwehlten Leib",
"Ein lebendige Rippe /",
"Nach dir ist mir /",
"Gratiosa cœli rosa Kranck vnd glümmet",
"Mein Hertz / durch Liebe verwundet.",
"Von Gott kompt mir ein Frewdenschein /",
"Wenn du mit deinen Eugelein /",
"Mich freundtlich thust anblicken /",
"O HERR Jesu mein trawtes Gut /",
"Dein Wort / dein Geist / dein Leib vnd Blut /",
"Mich innerlich erquicken. Nimm mich /",
"freundtlich Jn dein Arme /",
"Daß ich warme Werd von Gnaden /",
"Auff dein Wort komm ich geladen.",
"HERR Gott Vatter / mein starcker Heldt /",
"Du hast mich ewig / für der Welt /",
"In deinem Sohn geliebet /",
"Dein Sohn hat mich jhm selbst vertrawt /",
"Er ist mein Schatz / ich bin sein Braut /",
"Sehr hoch in jhm erfreuwet.",
"Eya Eya Himmlisch Leben /",
"wirdt er geben Mir dort oben /",
"Ewig soll mein Hertz jhn loben.",
"Zwingt die Sayten in Cythara.",
"Vnd laßt die süsse Musica,",
"Gantz frewdenreich erschallen:",
"Daß ich möge mit Jesulein /",
"Dem wunder schönen Bräutgam mein /",
"In stäter Liebe wallen.",
"Singet springet Jubilieret triumphieret /",
"Danckt dem HERREN Groß ist der König der Ehren.",
"Wie bin ich doch so hertzlich fro /",
"Daß mein Schatz ist das A vnd O /",
"Der Anfang / vnd das Ende:",
"Er wirdt mich doch zu seinem Preyß /",
"Auffnemmen in das Paradeiß /",
"Deß klopff ich in die Hände.",
"Amen Amen Komm du schone",
"FrewdenKrone Bleib du nicht lange /",
"Deiner wart ich mit Verlangen."
]

In [14]:
results

,sermon,verse,text_before,text_after
0,E000091,ey/ mein perle du werthe cron/,was ich gesucht bin ich gewehrt.,wahr gottes und marien sohn/
1,E000091,wahr gottes und marien sohn/,ey/ mein perle du werthe cron/,ein hochgeborner könig/ mein hertz heisst dich...
2,E000091,ein hochgeborner könig/ mein hertz heisst dich...,wahr gottes und marien sohn/,dein süsses evangelium/ ist lauter milch und h...
3,E000091,dein süsses evangelium/ ist lauter milch und h...,ein hochgeborner könig/ mein hertz heisst dich...,ey mein/ blümlein/ hosianna/
4,E000091,ey mein/ blümlein/ hosianna/,dein süsses evangelium/ ist lauter milch und h...,himlisch manna/ daß wir essen/
5,E000091,himlisch manna/ daß wir essen/,ey mein/ blümlein/ hosianna/,deiner kan ich nicht vergessen.
6,E000091,deiner kan ich nicht vergessen.,himlisch manna/ daß wir essen/,zwingt die seiten in cythara/
7,E000091,zwingt die seiten in cythara/,deiner kan ich nicht vergessen.,vnd lasst die süsse musica/
8,E000091,vnd lasst die süsse musica/,zwingt die seiten in cythara/,ganz freudenreich erschallen/ daß ich möge mit...
9,E000091,ganz freudenreich erschallen/ daß ich möge mit...,vnd lasst die süsse musica/,den allerhöchsten bräutgam mein/


In [15]:
def get_closest_verse(sermon_line: str, source: list) -> list:
    best_match = ""
    best_score = 0
    for verse in source:
        sim_score = fuzz.ratio(sermon_line, verse.lower())
        if sim_score > best_score:
            best_score = sim_score
            best_match = verse
    
    return [best_match, best_score, source.index(best_match)]

In [16]:
results[["best_match", "sim_score", "line"]] = results["verse"].apply(lambda x: pd.Series(get_closest_verse(x, liedtext_morgenstern)))

In [17]:
results

,sermon,verse,text_before,text_after,best_match,sim_score,line
0,E000091,ey/ mein perle du werthe cron/,was ich gesucht bin ich gewehrt.,wahr gottes und marien sohn/,Ey mein Perle / du werthe Kron /,90.322581,9
1,E000091,wahr gottes und marien sohn/,ey/ mein perle du werthe cron/,ein hochgeborner könig/ mein hertz heisst dich...,Wahr Gottes vnd Marien Sohn /,94.736842,10
2,E000091,ein hochgeborner könig/ mein hertz heisst dich...,wahr gottes und marien sohn/,dein süsses evangelium/ ist lauter milch und h...,"Mein Hertz heißt dich ein lilium,",68.131868,12
3,E000091,dein süsses evangelium/ ist lauter milch und h...,ein hochgeborner könig/ mein hertz heisst dich...,ey mein/ blümlein/ hosianna/,Jst lauter Milch vnd Honig /,60.759494,14
4,E000091,ey mein/ blümlein/ hosianna/,dein süsses evangelium/ ist lauter milch und h...,himlisch manna/ daß wir essen/,Ey mein Blümlein Hosianna /,94.545455,15
5,E000091,himlisch manna/ daß wir essen/,ey mein/ blümlein/ hosianna/,deiner kan ich nicht vergessen.,Himmlisch Manna Das wir essen /,91.803279,16
6,E000091,deiner kan ich nicht vergessen.,himlisch manna/ daß wir essen/,zwingt die seiten in cythara/,Deiner kan ich nicht vergessen.,100.000000,17
7,E000091,zwingt die seiten in cythara/,deiner kan ich nicht vergessen.,vnd lasst die süsse musica/,Zwingt die Sayten in Cythara.,89.655172,45
8,E000091,vnd lasst die süsse musica/,zwingt die seiten in cythara/,ganz freudenreich erschallen/ daß ich möge mit...,"Vnd laßt die süsse Musica,",90.566038,46
9,E000091,ganz freudenreich erschallen/ daß ich möge mit...,vnd lasst die süsse musica/,den allerhöchsten bräutgam mein/,Gantz frewdenreich erschallen:,62.790698,47


In [18]:
quoting_sermons = results['sermon'].values.tolist()

In [19]:
x = results.loc[results["best_match"] == "Singet springet Jubilieret triumphieret /"].values.tolist()[0]

In [21]:
lines_with_data = []
for line in liedtext_morgenstern:
    row_list = results.loc[results["best_match"] == line].values.tolist()
    lines_with_data.append({"name": line, "value":row_list})

lines_with_data

[{'name': 'Wie schön leuchtet der Morgenstern /',
  'value': [['E000061',
    'wie schön leuchtet der morgenstern.',
    'v. aus dem liede:',
    'das heutige heilige evangelium am 1.',
    'Wie schön leuchtet der Morgenstern /',
    95.77464788732395,
    0]]},
 {'name': 'Voll Gnad vnd Warheit von dem HERRN /', 'value': []},
 {'name': 'Die süsse Wurtzel Jesse?', 'value': []},
 {'name': 'Du Sohn Dauid auß Jacobs Stamm /', 'value': []},
 {'name': 'Mein König vnd mein Bräutigam /', 'value': []},
 {'name': 'Hast mir mein Hertz besessen /', 'value': []},
 {'name': 'Lieblich freundtlich Schön vnd herrlich /', 'value': []},
 {'name': 'Groß vnd ehrlich Reich von Gaben /', 'value': []},
 {'name': 'Hoch vnd sehr prächtig erhaben.', 'value': []},
 {'name': 'Ey mein Perle / du werthe Kron /',
  'value': [['E000091',
    'ey/ mein perle du werthe cron/',
    'was ich gesucht bin ich gewehrt.',
    'wahr gottes und marien sohn/',
    'Ey mein Perle / du werthe Kron /',
    90.32258064516128,
    9]

In [23]:
# when do quote types spike?

sermon = Sermon("E000036")

In [62]:
def year_helper(year):
    year_finder = re.compile(r'[0-9]{4}')
    year_cleaned = re.findall(year_finder, year)[0]
    return int(year_cleaned)

In [101]:
quotes_per_year = []
sermons_per_year = []
for id in ids:
    sermon_quotes = 0
    source_quotes = 0
    song_quotes = 0

    sermon = Sermon(id)

    year = year_helper(sermon.erscheinungsjahr)
    link = f'<a href="https://orgelpredigt.ur.de/{id}" target="_blank">{sermon.kurztitel}</a>'
    sermons_per_year.append([year, sermon.kurztitel, link])

    all_quotes = set(sermon.all_references)
    for quote in all_quotes:
        if quote.startswith("E00"):
            sermon_quotes += 1
        elif quote.startswith("E08") or quote.startswith("E09"):
            source_quotes += 1
        elif quote.startswith("E10"):
            song_quotes += 1
    quotes_per_year.append([year, [sermon_quotes, source_quotes, song_quotes]])

In [103]:
quotes_per_year

[[1602, [0, 7, 0]],
 [1605, [0, 1, 0]],
 [1606, [0, 8, 5]],
 [1610, [0, 3, 1]],
 [1621, [0, 13, 1]],
 [1624, [0, 23, 0]],
 [1628, [0, 6, 1]],
 [1647, [1, 20, 8]],
 [1648, [2, 11, 8]],
 [1651, [1, 24, 7]],
 [1652, [2, 15, 5]],
 [1660, [1, 10, 1]],
 [1664, [0, 7, 4]],
 [1666, [0, 11, 3]],
 [1667, [0, 10, 8]],
 [1671, [0, 15, 4]],
 [1672, [0, 5, 12]],
 [1673, [3, 10, 4]],
 [1675, [3, 6, 5]],
 [1676, [2, 12, 13]],
 [1676, [0, 14, 8]],
 [1680, [3, 10, 7]],
 [1681, [0, 15, 18]],
 [1683, [1, 10, 16]],
 [1685, [2, 12, 1]],
 [1686, [0, 32, 7]],
 [1687, [1, 45, 1]],
 [1689, [0, 24, 1]],
 [1695, [1, 23, 3]],
 [1696, [1, 10, 0]],
 [1700, [0, 13, 10]],
 [1704, [0, 28, 3]],
 [1704, [2, 49, 3]],
 [1709, [1, 7, 1]],
 [1709, [0, 15, 5]],
 [1711, [2, 49, 15]],
 [1711, [0, 15, 2]],
 [1720, [2, 104, 5]],
 [1721, [0, 13, 0]],
 [1721, [3, 12, 11]],
 [1721, [2, 1, 1]],
 [1726, [0, 4, 5]],
 [1727, [0, 8, 6]],
 [1728, [0, 157, 8]],
 [1730, [0, 5, 6]],
 [1735, [0, 11, 17]],
 [1737, [0, 1, 6]],
 [1739, [2, 20, 3

In [93]:
sermons_per_year

[[1602,
  'Christliche Predigt (Tübingen 1602)',
  '<a href="https://orgelpredigt.ur.de/E000001" target="_blank">Christliche Predigt (Tübingen 1602)</a>'],
 [1605,
  'Musica instrumentalis (Meißen 1605)',
  '<a href="https://orgelpredigt.ur.de/E000002" target="_blank">Musica instrumentalis (Meißen 1605)</a>'],
 [1606,
  'Christliche Predigt (Tübingen 1606)',
  '<a href="https://orgelpredigt.ur.de/E000029" target="_blank">Christliche Predigt (Tübingen 1606)</a>'],
 [1610,
  'Elogium Organi Musici (Altenburg 1610)',
  '<a href="https://orgelpredigt.ur.de/E000030" target="_blank">Elogium Organi Musici (Altenburg 1610)</a>'],
 [1621,
  'Corona Templi (Nürnberg 1621)',
  '<a href="https://orgelpredigt.ur.de/E000099" target="_blank">Corona Templi (Nürnberg 1621)</a>'],
 [1624,
  'Vlmische Orgel Predigt (Ulm 1624)',
  '<a href="https://orgelpredigt.ur.de/E000003" target="_blank">Vlmische Orgel Predigt (Ulm 1624)</a>'],
 [1628,
  'Musica ecclesiastica (Stettin 1628)',
  '<a href="https://orgel

In [94]:
colors = {
    'Orgelpredigten': 'rgb(135, 44, 162)',
    'Musikwerke': 'rgb(192, 54, 157)',
    'Literatur': 'rgb(234, 79, 136)'
}

In [95]:
df = pd.DataFrame(sermons_per_year, columns=["year", "title", "id"])
# Create full year range
all_years = pd.DataFrame({"year": range(1600,1801)})

# Merge to ensure every year appears, fill missing with 0
df_full = pd.merge(all_years, df, on="year", how="left").fillna(0)

agg = df.groupby("year").agg({
    "title": lambda x: "<br>".join(x.astype(str)),
    "id": lambda x: ", ".join(x.astype(str)),
    "year": "count"
}).rename(columns={"year": "count"}).reset_index()

In [99]:
fig = px.bar(agg, x='year', y='count',
             hover_data='id',
             color_discrete_sequence=['rgb(135, 44, 162)'])
fig.show()

In [102]:
df = pd.DataFrame(quotes_per_year, columns=["Jahr", "Anzahl"])
df[["Orgelpredigten", "Literatur", "Musikwerke"]] = pd.DataFrame(df["Anzahl"].tolist(), index=df.index)
df = df.drop(columns="Anzahl")

# Melt to long format for plotly express
df_long = df.melt(id_vars="Jahr", value_vars=["Orgelpredigten", "Literatur", "Musikwerke"],
                  var_name="type", value_name="Anzahl")

# Create figure
fig = go.Figure()

# Add line traces with custom colors
for t in df_long["type"].unique():
    subset = df_long[df_long["type"] == t]
    fig.add_trace(
        go.Scatter(
            x=subset["Jahr"],
            y=subset["Anzahl"],
            mode="lines+markers",
            name=t,
            line=dict(color=colors[t], width=2),
            marker=dict(size=8)
        )
    )

# Layout
fig.update_layout(
    title="Zeitliche Verteilung von Zitattypen",
    xaxis_title="Jahr",
    yaxis_title="Anzahl"
)

fig.show()

In [104]:
sermon = Sermon("E000036")

In [122]:
sermon.chunked[0][3]["types"]

['', '', '', '', '', ' bibel', ' bibel']

In [ ]:
t = ["", "", ""]

In [119]:
first_type = ""
for i in t:
    if (len(i) != 0):
        first_type = i
        break
print(first_type)

 .
